In [2]:
import openai


In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key="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")
MODEL = "openai/gpt-oss-120b"  # supports tool calling

# Quick connectivity check
ping = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
)
print(ping.choices[0].message.content)

Connected


In [5]:
from google.colab import userdata
from groq import Groq
from openai import OpenAI # Import OpenAI

# Get API keys
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") # OpenAI key is optional for fallback

# Initialize clients
groq_client = Groq(api_key=GROQ_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# Define models - primary is Groq, fallback is OpenAI
PRIMARY_MODEL = "llama3-8b-8192" # A common Groq model
FALLBACK_MODEL = "gpt-3.5-turbo" # A common OpenAI model

print(f"Primary LLM: Groq with model '{PRIMARY_MODEL}'")
if openai_client:
    print(f"Fallback LLM: OpenAI with model '{FALLBACK_MODEL}'")
else:
    print("OpenAI client not initialized (no OPENAI_API_KEY found). No fallback.")

# Quick connectivity check for primary client
try:
    ping = groq_client.chat.completions.create(
        model=PRIMARY_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
    )
    print(f"Groq primary client status: {ping.choices[0].message.content}")
except Exception as e:
    print(f"Groq primary client connection failed: {e}")

# If openai_client exists, perform a quick connectivity check for it too.
if openai_client:
    try:
        ping_openai = openai_client.chat.completions.create(
            model=FALLBACK_MODEL,
            messages=[{"role": "user", "content": "Reply with exactly: Connected"}],
        )
        print(f"OpenAI fallback client status: {ping_openai.choices[0].message.content}")
    except Exception as e:
        print(f"OpenAI fallback client connection failed: {e}")

Primary LLM: Groq with model 'llama3-8b-8192'
Fallback LLM: OpenAI with model 'gpt-3.5-turbo'
Groq primary client connection failed: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
OpenAI fallback client status: Connected


In [7]:
_GLOBAL_APPLICATION_LOGS = {
    "payment_gateway": [
        {"timestamp": "2026-08-23 14:02:11", "level": "INFO", "module": "checkout", "message": "Session initialized for card txn_98321."},
        {"timestamp": "2026-08-23 14:05:44", "level": "ERROR", "module": "stripe_api", "message": "Stripe connection timeout. HTTP status code 504 Gateway error."},
        {"timestamp": "2026-08-23 14:10:01", "level": "WARNING", "module": "fraud_check", "message": "High-risk velocity flag triggered for IP 192.168.1.45."}
    ],
    "inventory_manager": [
        {"timestamp": "2026-08-23 15:20:00", "level": "INFO", "module": "db_sync", "message": "Syncing stock counts for 1,450 active SKUs."},
        {"timestamp": "2026-08-23 15:22:15", "level": "ERROR", "module": "warehouse_service", "message": "Failed to update quantity for SKU_PRIME_09: Record Locked."},
    ],
    "user_auth": [
        {"timestamp": "2026-08-23 16:01:05", "level": "INFO", "module": "jwt_service", "message": "Token renewed successfully for uid_4402."},
        {"timestamp": "2026-08-23 16:45:30", "level": "CRITICAL", "module": "active_directory", "message": "LDAP Primary controller unresponsive fallback initiated."}
    ]
}

def fetch_application_logs(app_name: str, level: str = None, search_keyword: str = None) -> dict:
    app_key = app_name.lower().strip().replace(" ", "_")

    # Validation boundary check
    if app_key not in _GLOBAL_APPLICATION_LOGS:
        return {
            "status": "error",
            "message": f"Application repository '{app_name}' not found.",
            "available_applications": list(_GLOBAL_APPLICATION_LOGS.keys())
        }

    logs = _GLOBAL_APPLICATION_LOGS[app_key]

    if level:
        logs = [log for log in logs if log["level"].upper() == level.upper()]

    if search_keyword:
        kw = search_keyword.lower()
        logs = [log for log in logs if kw in log["message"].lower() or kw in log.get("module", "").lower()]

    return {
        "status": "success",
        "application": app_key,
        "records_found": len(logs),
        "data": logs
    }

In [8]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "fetch_application_logs",
            "description": "Universally queries, extracts, and parses system log records from any target corporate application context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "app_name": {
                        "type": "string",
                        "description": "The exact name or identifier of the application registry (e.g., 'payment_gateway', 'inventory_manager', 'user_auth')."
                    },
                    "level": {
                        "type": "string",
                        "enum": ["INFO", "WARNING", "ERROR", "CRITICAL"],
                        "description": "Optional level tag to filter events by severity importance profile."
                    },
                    "search_keyword": {
                        "type": "string",
                        "description": "Optional search term or pattern to narrow down files by message contents like 'timeout', 'SKU', or 'token'."
                    }
                },
                "required": ["app_name"],
            },
        },
    }
]

TOOL_IMPL = {
    "fetch_application_logs": fetch_application_logs,
}

In [9]:
SYSTEM_PROMPT = """
You are Log Fetcher, an advanced systems diagnostic IT agent.
- You diagnose multi-application stability by fetching logs.
- Clearly present diagnostic findings to users concisely. Keep entries factual.
- Keep final text responses completely clear and limited to 4 sentences max.
"""

In [13]:
import json

def call_llm(client_obj, model_name, messages, tools):
    return client_obj.chat.completions.create(
        model=model_name, messages=messages, tools=tools
    )

def call_model_with_retry(client_obj, model_name, messages, tools, max_retries=3):
    delay = 1.5
    for attempt in range(max_retries):
        try:
            return call_llm(client_obj, model_name, messages, tools)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"  [Error handling: {e} — retrying in {delay:.1f}s] using {model_name}")
            time.sleep(delay)
            delay *= 2

def run_log_fetcher(user_message, verbose=True, max_steps=6):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    current_client = groq_client
    current_model = PRIMARY_MODEL

    for step in range(max_steps):
        response = None
        try:
            response = call_model_with_retry(current_client, current_model, messages, TOOLS)
        except Exception as e_primary:
            print(f"Primary Agent Module ({current_model}) failed: {e_primary}")
            if openai_client:
                print(f"Triggering backup failover to OpenAI Layer ({FALLBACK_MODEL})...")
                current_client = openai_client
                current_model = FALLBACK_MODEL
                try:
                    response = call_model_with_retry(current_client, current_model, messages, TOOLS)
                except Exception as e_fallback:
                    return f"[System Engine Fault: {e_fallback}]"
            else:
                return "[Fatal Connection Error: No active processing layers reachable.]"

        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[Final Diagnostics Report]\n{msg.content}\n")
            return msg.content

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            if verbose:
                print(f"[Invoking Log Engine Tool] {name}({tool_call.function.arguments})")

            try:
                args = json.loads(tool_call.function.arguments)
                if name not in TOOL_IMPL:
                    raise ValueError(f"Unknown system module operation: {name}")
                result = TOOL_IMPL[name](**args)
            except Exception as e:
                result = {"error": str(e)}

            if verbose:
                print(f"[Log Streams Extracted Data Capture] {result}\n")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

    return "[Agent runtime steps tracking exhausted without reaching clean output baseline]"

In [14]:
print("--- TEST CASE 1: Querying Errors from Stripe APIs on Payment Processing App ---")
run_log_fetcher("Our billing checkout has issues. Pull up the payment gateway app logs containing 'ERROR'.")

print("--- TEST CASE 2: Text Keyword Content Filter Extraction On Inventory App ---")
run_log_fetcher("Can you look into our inventory manager utility logs to see if there are any mentions of 'SKU' entries?")

--- TEST CASE 1: Querying Errors from Stripe APIs on Payment Processing App ---
  [Error handling: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}} — retrying in 1.5s] using llama3-8b-8192
Primary Agent Module (llama3-8b-8192) failed: name 'time' is not defined
Triggering backup failover to OpenAI Layer (gpt-3.5-turbo)...
[Invoking Log Engine Tool] fetch_application_logs({"app_name":"payment_gateway","level":"ERROR"})
[Log Streams Extracted Data Capture] {'status': 'success', 'application': 'payment_gateway', 'records_found': 1, 'data': [{'timestamp': '2026-08-23 14:05:44', 'level': 'ERROR', 'module': 'stripe_api', 'message': 'Stripe connection timeout. HTTP status code 504 Gateway error.'}]}

[Final Diagnostics Report]
I found 1 error log 

'In the inventory manager logs:\n1. An INFO record on 2026-08-23 at 15:20:00 mentioned syncing stock counts for 1,450 active SKUs.\n2. An ERROR record on the same day at 15:22:15 reported a failure to update the quantity for SKU_PRIME_09 due to a Record Locked issue.'